<a href="https://colab.research.google.com/github/dharmika-p/PeerMatching/blob/main/PeerMatching_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import random

names = [
    "Alice","Bob","Charlie","David","Eva","Frank","Grace","Henry","Ivy","Jack",
    "Kiran","Latha","Manoj","Nina","Omar","Priya","Rahul","Sneha","Tarun","Uma",
    "Vikram","Waseem","Xavier","Yamini","Zara"
]

skills_list = [
    "python", "machine learning", "data science", "deep learning",
    "java", "spring boot", "react", "nodejs",
    "cloud computing", "cyber security", "nlp", "computer vision"
]

interests_list = [
    "ai research", "web development", "data analytics",
    "full stack", "mobile apps", "blockchain",
    "robotics", "gaming", "automation", "iot"
]

levels = ["beginner", "intermediate", "advanced"]

data = []

for i in range(300):
    name = random.choice(names) + str(i)


    skills = " ".join(random.sample(skills_list, random.randint(2, 4)))

    interests = " ".join(random.sample(interests_list, random.randint(1, 3)))

    level = random.choice(levels)

    data.append([name, skills, interests, level])

df = pd.DataFrame(data, columns=["name", "skills", "interests", "level"])

df.to_csv("users.csv", index=False)

print("Realistic 300-user dataset created!")

Realistic 300-user dataset created!


In [3]:
from google.colab import files
files.download('users.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
from google.colab import files
uploaded = files.upload()

Saving users.csv to users (1).csv


In [5]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler

In [8]:
# Phase 1: Load dataset and visualize basic structure
data = pd.read_csv("users.csv")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)

print("First 5 rows:")
print(data.head())

print("\nDataset Shape:", data.shape)

First 5 rows:
      name                                     skills                         interests         level
0   David0                     python computer vision                        automation      advanced
1  Vikram1  cloud computing java machine learning nlp                   web development      beginner
2   Priya2              machine learning data science  full stack automation blockchain  intermediate
3   Grace3        nlp cloud computing computer vision                iot data analytics      advanced
4     Ivy4                 spring boot cyber security                        blockchain  intermediate

Dataset Shape: (300, 4)


In [10]:
# Phase 2: Clean and prepare text data using NLP techniques

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

data['skills'] = data['skills'].apply(preprocess)
data['interests'] = data['interests'].apply(preprocess)

data['combined'] = data['skills'] + " " + data['interests']

display(data[['name', 'combined']].head())

,name,combined
0,David0,python computer vision automation
1,Vikram1,cloud computing java machine learning nlp web development
2,Priya2,machine learning data science full stack automation blockchain
3,Grace3,nlp cloud computing computer vision iot data analytics
4,Ivy4,spring boot cyber security blockchain


In [11]:
# Phase 3: Convert text to numerical features and add experience level

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(data['combined'])

level_map = {"beginner": 1, "intermediate": 2, "advanced": 3}
data['level_num'] = data['level'].map(level_map)

scaler = MinMaxScaler()
level_scaled = scaler.fit_transform(data[['level_num']])

features = np.hstack((tfidf_matrix.toarray(), level_scaled))

print("Feature Matrix Shape:", features.shape)

Feature Matrix Shape: (300, 32)


In [23]:
# Phase 4: Train KNN model for similarity-based recommendation

model_cosine = NearestNeighbors(metric='cosine')
model_cosine.fit(features)

model_euclidean = NearestNeighbors(metric='euclidean')
model_euclidean.fit(features)

print("Both Cosine and Euclidean models trained successfully!")

Both Cosine and Euclidean models trained successfully!


In [30]:
from IPython.display import display
import pandas as pd

# Phase 5: Generate peer recommendations (clean + explainable)

def recommend_peers(user_name, k=5):
    if user_name not in data['name'].values:
        print("User not found!")
        return

    user_index = data[data['name'] == user_name].index[0]

    # ---------- COSINE SIMILARITY ----------
    distances_c, indices_c = model_cosine.kneighbors(
        [features[user_index]], n_neighbors=k+1
    )

    cosine_results = []

    for i in range(1, len(indices_c[0])):
        idx = indices_c[0][i]
        similarity = 1 - distances_c[0][i]

        # Find common skills/interests
        user_words = set(data.iloc[user_index]['combined'].split())
        peer_words = set(data.iloc[idx]['combined'].split())
        common_words = list(user_words.intersection(peer_words))[:3]

        cosine_results.append({
            "Name": data.iloc[idx]['name'],
            "Skills": data.iloc[idx]['skills'],
            "Level": data.iloc[idx]['level'],
            "Score": round(similarity, 2),
            "Common Skills": ", ".join(common_words)
        })

    print("\n🔹 Cosine Similarity Results:\n")
    display(pd.DataFrame(cosine_results))

    # ---------- EUCLIDEAN DISTANCE ----------
    distances_e, indices_e = model_euclidean.kneighbors(
        [features[user_index]], n_neighbors=k+1
    )

    euclidean_results = []

    for i in range(1, len(indices_e[0])):
        idx = indices_e[0][i]

        euclidean_results.append({
            "Name": data.iloc[idx]['name'],
            "Skills": data.iloc[idx]['skills'],
            "Level": data.iloc[idx]['level'],
            "Distance": round(distances_e[0][i], 2)
        })

    print("\n🔹 Euclidean Distance Results:\n")
    display(pd.DataFrame(euclidean_results))

In [33]:
# Existing user recommendation
recommend_peers("David0")


🔹 Cosine Similarity Results:



,Name,Skills,Level,Score,Common Skills
0,Eva105,deep learning computer vision,advanced,0.85,"computer, automation, vision"
1,Ivy44,computer vision machine learning,advanced,0.85,"computer, automation, vision"
2,Rahul51,java computer vision,advanced,0.83,"computer, automation, vision"
3,Frank172,python computer vision,advanced,0.82,"computer, python, vision"
4,Uma208,data science computer vision machine learning python,intermediate,0.79,"computer, automation, vision"



🔹 Euclidean Distance Results:



,Name,Skills,Level,Distance
0,Eva105,deep learning computer vision,advanced,0.76
1,Ivy44,computer vision machine learning,advanced,0.78
2,Rahul51,java computer vision,advanced,0.83
3,Frank172,python computer vision,advanced,0.85
4,Uma208,data science computer vision machine learning python,intermediate,0.87


In [34]:
from IPython.display import display
import pandas as pd

# Phase 6: Recommendation for a NEW user (clean table output)

def recommend_new_user(skills, interests, level, k=5):

    # Preprocess input
    new_text = preprocess(skills + " " + interests)
    new_vector = vectorizer.transform([new_text]).toarray()

    # Convert level
    level_map = {"beginner": 1, "intermediate": 2, "advanced": 3}
    level_val = level_map[level]
    level_scaled = level_val / 3

    # Combine features
    new_features = np.hstack((new_vector, [[level_scaled]]))

    # ---------- COSINE ----------
    distances_c, indices_c = model_cosine.kneighbors(new_features, n_neighbors=k)

    cosine_results = []

    for i in range(len(indices_c[0])):
        idx = indices_c[0][i]
        similarity = 1 - distances_c[0][i]

        cosine_results.append({
            "Name": data.iloc[idx]['name'],
            "Skills": data.iloc[idx]['skills'],
            "Score": round(similarity, 2)
        })

    print("\n🔹 New User Recommendations (Cosine):\n")
    display(pd.DataFrame(cosine_results))

    # ---------- EUCLIDEAN ----------
    distances_e, indices_e = model_euclidean.kneighbors(new_features, n_neighbors=k)

    euclidean_results = []

    for i in range(len(indices_e[0])):
        idx = indices_e[0][i]

        euclidean_results.append({
            "Name": data.iloc[idx]['name'],
            "Skills": data.iloc[idx]['skills'],
            "Distance": round(distances_e[0][i], 2)
        })

    print("\n🔹 New User Recommendations (Euclidean):\n")
    display(pd.DataFrame(euclidean_results))

In [35]:
# New user recommendation
recommend_new_user(
    skills="python machine learning",
    interests="data science",
    level="intermediate"
)


🔹 New User Recommendations (Cosine):



,Name,Skills,Score
0,Yamini113,data science machine learning nodejs python,0.84
1,Uma208,data science computer vision machine learning python,0.82
2,Nina210,python data science,0.79
3,Rahul270,machine learning data science react nodejs,0.76
4,Latha187,data science java python,0.74



🔹 New User Recommendations (Euclidean):



,Name,Skills,Distance
0,Uma208,data science computer vision machine learning python,0.70
1,Yamini113,data science machine learning nodejs python,0.78
2,Nina210,python data science,0.87
3,Priya2,machine learning data science,0.90
4,Rahul270,machine learning data science react nodejs,0.92
